In [1]:
import os
import zipfile
import json
import pickle
import hashlib
import pandas as pd
from fingerprint_parser.shell_attributes_parser import ShellAttributeParser
from fingerprint_parser.sdk_attributes_parser import SdkAttributeParser
from fingerprint_parser.cp_attributes_parser import CpAttributeParser

INPUT_DATA_DIR = "DUMMY_DATA"
PREPARE_DIR = "DUMMY_DATA_PREPARED"
STRUCTURE_DIR = "DUMMY_DATA_STRUCTURE"

In [2]:
def load_json_file(file_path):
    """Helper function to load JSON data from a file."""
    with open(file_path, 'r') as file:
        return json.load(file)
    
def save_json_file(data, file_path):
    """Helper function to save JSON data to a file."""
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)
          
# Hash the sdk structure
def hash_sdk_structure(l):
    return hashlib.sha256(pickle.dumps(l)).hexdigest()

def check_device_virtual(data):
    """
    Check if the device is virtual (emulator or virtual machine) based on build properties.
    """
    is_device_virtual = False

    # Convert properties to lowercase for case-insensitive checks
    manufacturer = str(data.get("android.os.Build.MANUFACTURER", "")).lower()
    model = str(data.get("android.os.Build.MODEL", "")).lower()
    hardware = str(data.get("android.os.Build.HARDWARE", "")).lower()
    fingerprint = str(data.get("android.os.Build.FINGERPRINT", "")).lower()
    product = str(data.get("android.os.Build.PRODUCT", "")).lower()
    board = str(data.get("android.os.Build.BOARD", "")).lower()
    brand = str(data.get("android.os.Build.BRAND", "")).lower()
    device = str(data.get("android.os.Build.DEVICE", "")).lower()
    kernel = str(data.get("kernel_information", "")).lower()
    system_logs = str(data.get("system_logs", "")).lower()
    
    virtual_flags = ["vbox", "virtual", "qemu", "vmware", "hypervisor", "kvm", "xen", "bochs", "nox"]
    # Check conditions for emulator or virtual machine
    is_emulator = (
        "genymotion" in manufacturer
        or "google_sdk" in model
        or "droid4x" in model
        or "emulator" in model
        or "android sdk built for x86" in model
        or hardware == "goldfish"
        or hardware == "vbox86"
        or "nox" in hardware
        or fingerprint.startswith("generic")
        or product in ["sdk", "google_sdk", "sdk_x86", "vbox86p"]
        or "nox" in product
        or "nox" in board
        or (brand.startswith("generic") and device.startswith("generic"))
        or "x86" in kernel 
        or "amd64" in kernel
        or any(flag in system_logs for flag in virtual_flags)
    )

    is_device_virtual = is_emulator

    return is_device_virtual

In [3]:
def preapre_fingerprint(data, filename):
    cleaned_data = {}
    is_incomplete = True
    suffix = ".ANDROID_ID"
    uuid,timestamp = filename.split("_")
    timestamp = int(timestamp.split(".")[0])
    sdk_structure = set()
    for item in data:
        # Check the first (and only) key-value pair in the dictionary
        for key, value in item.items():
            # Mark as complete if a key ends with the required suffix
            if key.endswith(suffix):
                is_incomplete = False
                
            # Parse shell attributes
            if ShellAttributeParser.isShellAttribute(key):
                value = ShellAttributeParser.parse(key, value, timestamp)
                # add it to cleaned data
                if value : 
                    if key == "ringtones_list_ext":
                        key = "ringtones_list"
                    # Flatten if is a dict
                    if isinstance(value, dict):
                        for k,v in value.items():
                            cleaned_data[f"{key}.{k}"] = v
                    elif isinstance(value, list):
                        if len(value)> 0 and len(value) == 1:
                            cleaned_data[key] = value[0]
                        elif len(value)>0 : 
                            cleaned_data[key] = value
                    else: 
                        cleaned_data[key] = value
            elif SdkAttributeParser.isSdkAttribute(key):
                # For SDK attributes, keep the nbSdk and SDK Structure
                sdk_structure.add(key)
                # Parse using SdkAttributeParser
                value = SdkAttributeParser.parse(key, value)
                # add it to cleaned data
                if value :
                    # Flatten if is a dict
                    if isinstance(value, dict):
                        for k,v in value.items():
                            cleaned_data[f"{key}.{k}"] = v
                    elif isinstance(value, list):
                        if len(value)> 0 and len(value) == 1:
                            cleaned_data[key] = value[0]
                        elif len(value)>0 : 
                            cleaned_data[key] = value
                    else: 
                        cleaned_data[key] = value
            elif CpAttributeParser.isCpAttribute(key):
                value = CpAttributeParser.parse(key,value)
                if value: 
                    # Flatten if is a dict
                    if isinstance(value, dict):
                        for k,v in value.items():
                            cleaned_data[f"{key}.{k}"] = v
                    elif isinstance(value, list):
                        if len(value)> 0 and len(value) == 1:
                            cleaned_data[key] = value[0]
                        elif len(value)>0 : 
                            cleaned_data[key] = value
                    else: 
                        cleaned_data[key] = value
            else: 
                cleaned_data[key] = value
            break # contains only one key-value pair in each dictionary
    # Return empty list if no complete fingerprints are found
    if is_incomplete : 
        return {}
    
    sdk_structure = sorted(list(sdk_structure))
    cleaned_data["structureSdk"] = hash_sdk_structure(sdk_structure)
    cleaned_data["nbSdk"] = len(sdk_structure)
    cleaned_data["timestamp"] = timestamp
    cleaned_data["uuid"] = uuid
    cleaned_data["isDeviceVirtual"] = check_device_virtual(cleaned_data)
    if "isDeviceRooted" not in cleaned_data:
        cleaned_data["isDeviceRooted"] = "unknown"
    if "isDeveloperModeEnabled" not in cleaned_data:
        cleaned_data["isDeveloperModeEnabled"] = -1 # means unknown
    
    # save the sdk structure 
    structure_file_path = os.path.join(STRUCTURE_DIR,filename)     
    save_json_file(sdk_structure,structure_file_path)
    
    return cleaned_data

In [4]:
def extract_and_clean_archives(directory):
    """
    Extract archives found in the specified directory, clean the JSON data,
    and save the cleaned data to new files if they haven't been processed already.
    """
    os.makedirs(PREPARE_DIR, exist_ok=True)
    os.makedirs(STRUCTURE_DIR, exist_ok=True)
    
    min_cleaned_data_size = float('inf')
    min_data_size = float('inf')
    max_cleaned_data_size = 0
    max_data_size = 0
    
    # Iterate through the files in the directory
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)

        # Check if the file is a zip archive
        if filename.endswith('.zip'):
            cleaned_file_name = f"{filename.split('.')[0]}.json"
            cleaned_file_path = os.path.join(PREPARE_DIR, cleaned_file_name)
            
            # Skip if the cleaned file already exists
            if os.path.exists(cleaned_file_path):
                print(f"Skipping '{filename}' as it is already processed.")
                continue
            
            # Extract the archive
            with zipfile.ZipFile(file_path, 'r') as archive:
                # Check if data.json is in the archive
                if 'data.json' in archive.namelist():
                    archive.extract('data.json', directory)

                    # Load and clean the JSON data
                    json_file_path = os.path.join(directory, 'data.json')
                    with open(json_file_path, 'r') as json_file:
                        data = json.load(json_file)
                        cleaned_data = preapre_fingerprint(data, cleaned_file_name)
                        if cleaned_data:
                            min_data_size = min(min_data_size, len(data))
                            max_data_size = max(max_data_size, len(data))
                            min_cleaned_data_size = min(min_cleaned_data_size, len(cleaned_data))
                            max_cleaned_data_size = max(max_cleaned_data_size, len(cleaned_data))

                            # Save the cleaned JSON data with a new name
                            with open(cleaned_file_path, 'w') as cleaned_file:
                                json.dump(cleaned_data, cleaned_file, indent=4)
                            
                            print(f"Processed '{filename}' and saved cleaned data")
                    
                    # Remove the extracted data.json file
                    os.remove(json_file_path)
    
    print("max data size :", max_data_size) 
    print("min data size :", min_data_size)
    print("max cleaned data size :", max_cleaned_data_size) 
    print("min cleaned data size :", min_cleaned_data_size)


In [5]:
extract_and_clean_archives(INPUT_DATA_DIR)

Processed '7e5ab2b2-6afd-444d-821c-166779bf1e0b_1773071911657.zip' and saved cleaned data
Processed 'a1ce5235-0003-432f-b97f-c520d33b4f0b_1773065822030.zip' and saved cleaned data
Processed 'cc78c680-dc8c-483a-9917-9e145c0776a1_1773065607266.zip' and saved cleaned data
Processed '1afe3e42-8731-4c44-9e09-ebd276ed0e62_1773071511790.zip' and saved cleaned data
Processed '60b15af1-1101-458f-a6a2-be60b0986f7a_1773064217128.zip' and saved cleaned data
Processed '5337d73d-8e94-4c85-94d8-ff54245bb62c_1773065040077.zip' and saved cleaned data
Processed '62f3d8db-357d-468b-9774-4207e479d12c_1773071118299.zip' and saved cleaned data
Processed '3170ad9f-d5fd-48e5-9cf0-f271a14a1cc4_1773064820592.zip' and saved cleaned data
Processed 'c944fe7c-6900-4412-b8ef-a773d689be3f_1773068740515.zip' and saved cleaned data
Processed 'cc78c680-dc8c-483a-9917-9e145c0776a1_1773065425131.zip' and saved cleaned data
Processed 'c944fe7c-6900-4412-b8ef-a773d689be3f_1773068552483.zip' and saved cleaned data
Processed 

In [6]:
def extract_info_fingerprint(data, filename):
    info = {
        "filename": filename,
        "manufacturer": data.get("android.os.Build.MANUFACTURER", "unknown"),
        "model": data.get("android.os.Build.MODEL", "unknown"),
        "sdk_version": data.get("android.os.Build.VERSION.SDK_INT", "unknown"),
        "android_id": data.get("content://settings/secure.android_id", "unknown"),
        "uuid": data.get("uuid", "unknown"),
        "timestamp": data.get("timestamp", "unknown"),
        "isDeviceRooted": data.get("isDeviceRooted", "unknown"),
        "isDeviceVirtual": data.get("isDeviceVirtual", "unknown"),
        "security_patch": data.get("android.os.Build.VERSION.SECURITY_PATCH",""),
        "firmware_version": data.get("android.os.Build.getRadioVersion", "unknown"),
        "networkCountryIso": data.get("android.telephony.TelephonyManager.getNetworkCountryIso", "unknown")
    }
    
    return info
    
def extract_info_directory(input_dir,output_file_name):
    # Read the csv file 
    #df = pd.read_csv(output_file_name)
    
    df = pd.DataFrame(columns= ["filename",
        "manufacturer",
        "model",
        "sdk_version",
        "android_id",
        "uuid",
        "timestamp",
        "isDeviceRooted",
        "isDeviceVirtual",
        "security_patch", 
        "firmware_version",
        "networkCountryIso"])
    
    # Iterate through the files in the directory
    for filename in os.listdir(input_dir):
        file_path = os.path.join(input_dir, filename)

        # Check if the file is a JSON file with the expected format
        if filename.endswith('.json'):
            if filename not in df["filename"].values:
                data = load_json_file(file_path)
                info = extract_info_fingerprint(data, filename)
                row_df = pd.DataFrame([info])
                df = pd.concat(
                    [df, row_df],
                    ignore_index=True
                )
                print(f"Adding fingerprint file {filename} stats")                                     

    df.to_csv(output_file_name, index=False)
    

In [7]:
extract_info_directory(PREPARE_DIR,"about_fingerprints.csv")

Adding fingerprint file 3170ad9f-d5fd-48e5-9cf0-f271a14a1cc4_1773064820592.json stats
Adding fingerprint file 62f3d8db-357d-468b-9774-4207e479d12c_1773071118299.json stats
Adding fingerprint file 1afe3e42-8731-4c44-9e09-ebd276ed0e62_1773071323547.json stats
Adding fingerprint file c944fe7c-6900-4412-b8ef-a773d689be3f_1773068740515.json stats
Adding fingerprint file 62f3d8db-357d-468b-9774-4207e479d12c_1773070931424.json stats
Adding fingerprint file 98a91d07-cae6-4f7d-8449-b653f275334a_1773063784296.json stats
Adding fingerprint file c944fe7c-6900-4412-b8ef-a773d689be3f_1773068552483.json stats
Adding fingerprint file 1afe3e42-8731-4c44-9e09-ebd276ed0e62_1773071511790.json stats
Adding fingerprint file cc78c680-dc8c-483a-9917-9e145c0776a1_1773065425131.json stats
Adding fingerprint file 3170ad9f-d5fd-48e5-9cf0-f271a14a1cc4_1773064638056.json stats
Adding fingerprint file a1ce5235-0003-432f-b97f-c520d33b4f0b_1773065822030.json stats
Adding fingerprint file cc78c680-dc8c-483a-9917-9e145c

In [10]:
# Load csv file containing metadata of all fingerprints
fingerprints_df = pd.read_csv("about_fingerprints.csv")
print(fingerprints_df.shape)
# Sort fingerprints by android_id and timestamp
fingerprints_latest_df = fingerprints_df.sort_values(["android_id", "timestamp"])

# Interval in days
max_interval_per_id_seconds = (
    fingerprints_latest_df.groupby("android_id")["timestamp"]
      .apply(lambda s: s.diff().max() / 1000)
)
max_interval_per_id_seconds = max_interval_per_id_seconds.round(2)
max_interval_per_id_seconds = max_interval_per_id_seconds.dropna()
max_interval_per_id_seconds = max_interval_per_id_seconds.sort_values(ascending=False)

print("Fingerprints statistics")
print(f"Total Fingerprints          : {fingerprints_latest_df["filename"].nunique()}")
print(f"Unique device               : {fingerprints_latest_df["android_id"].nunique()}")
print(f"Max time between 2 scans    : {max_interval_per_id_seconds.max()} seconds")
print(f"Min time between 2 scans    : {max_interval_per_id_seconds.min()} seconds")
print(f"Mean time between 2 scans   : {max_interval_per_id_seconds.mean()} seconds")
print(f"Median time between 2 scans : {max_interval_per_id_seconds.median()} seconds")

(22, 12)
Fingerprints statistics
Total Fingerprints          : 22
Unique device               : 11
Max time between 2 scans    : 191.62 seconds
Min time between 2 scans    : 180.76 seconds
Mean time between 2 scans   : 185.63454545454547 seconds
Median time between 2 scans : 186.88 seconds
